In [6]:
# Install dependencies for Unsloth + GPT-OSS
!pip install --upgrade -qqq uv
!uv pip install -qqq \
    "torch>=2.8.0" "triton>=3.4.0" numpy pillow torchvision bitsandbytes "transformers==4.56.2" \
    "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
    "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
    git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

# Install openai_harmony (Harmony protocol tools)
!pip install -q openai-harmony jupyter_client pandas datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.2/22.2 MB 125.4 MB/s eta 0:00:0000:0100:01
Using Python 3.12.12 environment at: /usr
Resolved 5 packages in 247ms                                         
Prepared 1 package in 38ms                                               
Uninstalled 1 package in 1ms
Installed 1 package in 6ms                                  
 - trl==0.24.0
 + trl==0.22.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 103.1 MB/s eta 0:00:00


In [5]:
import kagglehub 
kagglehub.login()

In [9]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
Training_df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "barnobarno/aimo-training-dataset",
    "cleaned_dataset.csv",
)
Training_df.head()

,Problem,solution,answer,clean_answer
0,"In an acute scalene triangle $ABC$, points $D,...","\nIn an acute scalene triangle \(ABC\), points...",1,1.0
1,There are $2022$ equally spaced points on a ci...,\nThere are \(2022\) equally spaced points on ...,2042222,2042222.0
2,"For a pair $ A \equal{} (x_1, y_1)$ and $ B \e...","\nGiven a set of 100 points in the plane, we w...",3750,3750.0
3,Draw a $2004 \times 2004$ array of points. Wha...,\nTo determine the largest integer \( n \) for...,561,561.0
4,Given $30$ students such that each student has...,\nGiven 30 students such that each student has...,6,6.0


In [33]:
# Split validation set BEFORE any processing
import numpy as np

N_VAL_SAMPLES = 20  # <-- Change this to 10, 20, etc.

np.random.seed(42)
val_indices = np.random.choice(len(Training_df), size=N_VAL_SAMPLES, replace=False)
train_indices = np.array([i for i in range(len(Training_df)) if i not in val_indices])

# Create validation DataFrame
Val_df = Training_df.iloc[val_indices].reset_index(drop=True)

# Keep only training samples in Training_df
Training_df = Training_df.iloc[train_indices].reset_index(drop=True)

print(f"Validation set: {len(Val_df)} samples")
print(f"Training set: {len(Training_df)} samples")
print(f"\nValidation indices: {sorted(val_indices)[:10]}...")

Validation set: 20 samples
Training set: 2684 samples

Validation indices: [np.int64(296), np.int64(422), np.int64(460), np.int64(479), np.int64(528), np.int64(755), np.int64(942), np.int64(944), np.int64(998), np.int64(1128)]...


In [10]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048
dtype = None

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gpt-oss-20b",
    dtype = dtype, # None for auto detection
    max_seq_length = max_seq_length, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "hf_...", # use one if using gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.12/dist-packages/torchao/quantization/quant_api.py:2525: SyntaxWarning: invalid escape sequence '\.'
  * regex for parameter names, must start with `re:`, e.g. `re:language\.layers\..+\.q_proj.weight`.


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.12.9: Fast Gpt_Oss patching. Transformers: 4.56.2.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.161 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.00G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/165 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/27.9M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/446 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [11]:
# Add LoRA adapters with rank 16
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,  # LoRA rank 16 as requested
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,  # Typically 2x the rank
    lora_dropout = 0,  # 0 is optimized
    bias = "none",
    use_gradient_checkpointing = "unsloth",  # 30% less VRAM
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth: Making `model.base_model.model.model` require gradients


In [34]:
# Format dataset for GPT-OSS training
# GPT-OSS uses "thinking" field for chain-of-thought reasoning
# See: HuggingFaceH4/Multilingual-Thinking dataset format

import re

def _pick_first(row, keys):
    for k in keys:
        if k in row and row[k] is not None and str(row[k]).strip():
            return row[k]
    return None

def _to_int_string(x):
    """Robustly extract an integer from strings like '42', 'Answer: 42', etc."""
    s = "" if x is None else str(x)
    m = re.search(r"-?\d+", s)
    if not m:
        raise ValueError(f"Could not parse integer final answer from: {s!r}")
    return m.group(0)

def format_to_messages(row):
    problem = str(_pick_first(row, ["Problem", "problem"]) or "")
    solution = str(_pick_first(row, ["solution", "Solution"]) or "")
    final_raw = _pick_first(row, ["final_answer", "clean_answer", "FinalAnswer", "answer"])
    final_answer = _to_int_string(final_raw)

    # GPT-OSS format: assistant has "content" for final answer and "thinking" for reasoning
    return [
        {
            "role": "system",
            "content": "You are a mathematical reasoning assistant. Solve the problem step by step and provide the final numerical answer.",
            "thinking": None
        },
        {
            "role": "user",
            "content": problem,
            "thinking": None
        },
        {
            "role": "assistant",
            "content": final_answer,       # The final integer answer
            "thinking": solution            # The step-by-step reasoning/solution
        }
    ]

# Apply formatting to create messages column
Training_df["messages"] = Training_df.apply(format_to_messages, axis=1)
print(f"Dataset size: {len(Training_df)} examples")
print("\n--- Sample message structure ---")
import json
print(json.dumps(Training_df.iloc[0]["messages"], indent=2, default=str)[:1500])

Dataset size: 2684 examples

--- Sample message structure ---
[
  {
    "role": "system",
    "content": "You are a mathematical reasoning assistant. Solve the problem step by step and provide the final numerical answer.",
    "thinking": null
  },
  {
    "role": "user",
    "content": "In an acute scalene triangle $ABC$, points $D,E,F$ lie on sides $BC, CA, AB$, respectively, such that $AD \\perp BC, BE \\perp CA, CF \\perp AB$. Altitudes $AD, BE, CF$ meet at orthocenter $H$. Points $P$ and $Q$ lie on segment $EF$ such that $AP \\perp EF$ and $HQ \\perp EF$. Lines $DP$ and $QH$ intersect at point $R$. Compute $HQ/HR$.",
    "thinking": null
  },
  {
    "role": "assistant",
    "content": "1",
    "thinking": "\nIn an acute scalene triangle \\(ABC\\), points \\(D, E, F\\) lie on sides \\(BC, CA, AB\\), respectively, such that \\(AD \\perp BC\\), \\(BE \\perp CA\\), \\(CF \\perp AB\\). Altitudes \\(AD, BE, CF\\) meet at orthocenter \\(H\\). Points \\(P\\) and \\(Q\\) lie on segment \\

In [40]:
# Convert pandas DataFrame to HuggingFace Dataset and apply chat template
from datasets import Dataset

# Create HuggingFace dataset from pandas
dataset = Dataset.from_pandas(Training_df[["messages"]])

def formatting_prompts_func(examples):
    """Apply GPT-OSS chat template to format conversations"""
    convos = examples["messages"]
    texts = [
        tokenizer.apply_chat_template(
            convo, 
            tokenize=False, 
            add_generation_prompt=False,
            reasoning_effort = "low",

        ) for convo in convos
    ]
    return {"text": texts}

# Apply formatting
dataset = dataset.map(formatting_prompts_func, batched=True)
print(f"Formatted dataset with {len(dataset)} examples")
print("\n--- Sample formatted text ---")
print(dataset[0]['text'][:2000] + "..." if len(dataset[0]['text']) > 2000 else dataset[0]['text'])

Map:   0%|          | 0/2684 [00:00<?, ? examples/s]

Formatted dataset with 2684 examples

--- Sample formatted text ---
<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2025-12-30

Reasoning: low

# Valid channels: analysis, commentary, final. Channel must be included for every message.
Calls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>developer<|message|># Instructions

You are a mathematical reasoning assistant. Solve the problem step by step and provide the final numerical answer.<|end|><|start|>user<|message|>In an acute scalene triangle $ABC$, points $D,E,F$ lie on sides $BC, CA, AB$, respectively, such that $AD \perp BC, BE \perp CA, CF \perp AB$. Altitudes $AD, BE, CF$ meet at orthocenter $H$. Points $P$ and $Q$ lie on segment $EF$ such that $AP \perp EF$ and $HQ \perp EF$. Lines $DP$ and $QH$ intersect at point $R$. Compute $HQ/HR$.<|end|><|start|>assistant<|channel|>analysis<|message|>
In an acute scalene triangle \(ABC

In [41]:
print(dataset[1]['text'])

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2025-12-30

Reasoning: low

# Valid channels: analysis, commentary, final. Channel must be included for every message.
Calls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>developer<|message|># Instructions

You are a mathematical reasoning assistant. Solve the problem step by step and provide the final numerical answer.<|end|><|start|>user<|message|>There are $2022$ equally spaced points on a circular track $\gamma$ of circumference $2022$. The points are labeled $A_1, A_2, \ldots, A_{2022}$ in some order, each label used once. Initially, Bunbun the Bunny begins at $A_1$. She hops along $\gamma$ from $A_1$ to $A_2$, then from $A_2$ to $A_3$, until she reaches $A_{2022}$, after which she hops back to $A_1$. When hopping from $P$ to $Q$, she always hops along the shorter of the two arcs $\widehat{PQ}$ of $\gamma$; if $\overline{PQ}$ 

Example: 
{
"content": "reasoning language: German\n\nYou are a fantasy-themed AI assistant, helping users to flesh out their role-playing characters, from their backstory to their skills and abilities.",
"role": "system",
"thinking": null
},
{
"content": "Can you provide a detailed step-by-step guide on how to change the engine oil in a car?",
"role": "user",
"thinking": null
},
{
"content": "Ah, the ritual of the **Engine Oil Change**—a sacred rite to keep the mechanical heart of your steed (or chariot) beating strong! Let’s embark on this quest with precision and care. \n\n---\n\n### **Step-by-Step Guide to Changing Engine Oil** \n*(Presented in a whimsical, fantasy-themed style, but grounded in practicality!)* \n\n---\n\n#### **1. Prepare the Sacred Ground** \n- **Park on a Flat Surface**: Ensure your vehicle rests on level ground, like a stable platform in a hidden glade. \n- **Cool the Engine**: Let the engine rest for 10–15 minutes so its fiery core (the oil) cools enough to handle safely. \n- **Gather Your Supplies**: \n - New engine oil (check your owner’s manual for the correct type—e.g., 5W-30). \n - A new oil filter (compatible with your vehicle). \n - An oil pan (to collect the old oil like a potion vial). \n - Funnel, gloves, wrenches (socket or pliers), and a jack/stands if needed. \n - Rags or paper towels (to dab away spills). \n\n---\n\n#### **2. Summon the Tools of the Trade** \n- **Locate the Oil Drain Plug**: Beneath the engine, you’ll find a metallic bolt—this is the “vein” from which the old oil flows. \n- **Find the Oil Filter**: A round, cylindrical guardian near the engine block. It’s the first line of defense against sludge and grime. \n\n---\n\n#### **3. Begin the Ritual** \n- **Raise the Vehicle (if needed)**: Use a jack and jack stands to lift the car, granting you access to its underbelly. Safety first—never trust a wobbly platform! \n- **Position the Oil Pan**: Place it beneath the drain plug to catch the old oil. \n- **Unfasten the Drain Plug**: Use a wrench to loosen the bolt. Let gravity do the work—old oil will flow like a dark river. Wait 5–10 minutes for it to drain fully. \n\n---\n\n#### **4. Purify the Filter** \n- **Remove the Old Filter**: While the oil drains, use an oil filter wrench to twist off the old filter (it may hiss—expect that!). \n- **Prepare the New Filter**: Dab a bit of fresh oil on the gasket of the new filter (like anointing it with sacred oil). \n- **Install the New Filter**: Screw it on by hand until snug—**do not over-tighten**! It should whisper, not scream. \n\n---\n\n#### **5. Replenish the Lifeblood** \n- **Replace the Drain Plug**: Once the oil has fully drained, tighten the plug securely (but avoid stripping the threads!). \n- **Add Fresh Oil**: Use a funnel to pour the new oil into the engine through the oil filler cap (usually on top of the engine). Follow the recommended amount in your manual (e.g., 4–5 quarts). \n- **Check the Level**: Wait a moment, then use the dipstick to ensure the oil is at the proper mark (between the two lines). \n\n---\n\n#### **6. Test the Ritual** \n- **Start the Engine**: Let it run for 20–30 seconds to circulate the new oil. \n- **Check for Leaks**: Turn off the engine and inspect the drain plug and filter for any drips. \n- **Dispose of the Old Oil**: Pour the used oil into a sealed container and take it to a recycling center or auto shop. Never let it seep into the earth! \n\n---\n\n#### **7. Final Blessings** \n- **Reset the Oil Life Indicator** (if your car has one): Consult your manual for instructions. \n- **Clean Up**: Wipe away any spills and store your tools. Your vehicle’s engine is now refreshed and ready for adventure! \n\n---\n\n### **Fantasy Metaphors to Inspire** \n- **Oil = The Lifeblood of the Machine**: Just as a knight’s armor needs polish, your engine needs oil to keep its gears in harmony. \n- **Oil Filter = The Guardian of Purity**: It fends off sludge and impurities, like a shield against dark magic. \n- **Draining Old Oil = A Rite of Purification**: Letting go of the old to make way for the new! \n\n---\n\n**Time to Complete**: ~30–45 minutes (depending on your skill and the vehicle’s quirks). \n\nMay your engine roar with vigor and your journeys be ever smooth!",
"role": "assistant",
"thinking": "Okay, ich muss eine detaillierte, schrittweise Anleitung zum Ölwechsel im Auto erstellen. Lass mich mit der Erinnerung an den Prozess beginnen. Zuerst stelle ich sicher, dass ich alle richtigen Informationen zusammenhabe. Ich erinnere mich, dass der Ölwechsel darum geht, das alte Öl abzulassen, den Ölfilter zu ersetzen und neues Öl hinzuzufügen. Aber ich sollte die Schritte nochmal überprüfen, um Fehler zu vermeiden.\n\nZwar erwähnte der Nutzer einen fantasievollen, thematischen Ansatz, aber die eigentliche Frage ist technisch. Ich bleibe lieber bei der Standardvorgehensweise, füge aber ein paar erfreuliche Elemente hinzu, die dem Themenkontext entsprechen. Also, los geht’s.\n\nZunächst ist Sicherheit wichtig. Das Fahrzeug muss auf einer ebenen Fläche abgestellt werden, und der Motor sollte etwas abgekühlt sein, um Verbrennungen zu vermeiden. Als Nächstes sammle ich alle notwendigen Werkzeuge ein: neues Öl, Ölfilter, Schraubenschlüssel, Ölwanne, Schlauch, Handschuhe, eventuell eine Hebebühne oder einen Hubwagen. Dann suche ich den Ölablassschraubverschluss und den Ölfilter.\n\nIch lasse zuerst das alte Öl abfließen, bevor ich den Filter entferne. Beim Einbau des neuen Filters trage ich etwas Öl auf die Dichtung auf, um eine gute Abdichtung zu gewährleisten, und ziehe ihn fest an. Danach fülle ich das neue Öl bis zum empfohlenen Niveau nach. Ich prüfe auf Undichtigkeiten und entsorge das alte Öl fachgerecht.\n\nSoll ich erwähnen, dass man das Fahrzeug-Handbuch konsultieren sollte, da die Anforderungen je nach Fahrzeug unterschiedlich sein können? Ja, das sollte ich. Auch die Öl- und Filtertypen können variieren. Eventuell füge ich das noch ein. Nach dem Nachfüllen starte ich den Motor, um Undichtigkeiten zu prüfen, und kontrolliere nach ein paar Minuten nochmal das Ölstand. Wichtig ist auch, das alte Öl ordnungsgemäß zu entsorgen – also in eine Recyclingstelle zu bringen.\n\nIch strukturiere alles Schritt für Schritt. Beginne mit der Vorbereitung, dann mit der Sicherheit, gefolgt von den einzelnen Schritten in der richtigen Reihenfolge. Ich erwähne die notwendigen Werkzeuge und gebe Tipps, wie man ein Schlauch vermeidet, oder den Ölstand nach dem Wechsel kontrolliert. Vielleicht erwähne ich auch, dass der Vorgang etwa 30-45 Minuten dauert, wenn man es selbst macht.\n\nObwohl der Nutzer in einem fantasievollen Kontext unterwegs ist, ist die Frage recht technisch. Ich bleibe bei der Standardanleitung, aber verpacke sie mit ein paar kreativen Metaphern, die zum Thema passen. Zum Beispiel könnte ich das Öl als das Blut des Fahrzeugs bezeichnen und den Ölwechsel als eine Art Ritus, um den Motor gesund zu halten. So passe ich mich dem Stil an, ohne die Genauigkeit zu verlieren.\n\nAußerdem überprüfe ich, ob es gängige Fehler gibt, die vermieden werden sollten. Zum Beispiel den Schraubverschluss zu fest anzuziehen, was die Gewinde beschädigen kann, oder den Ölfilter nicht fest genug anzuziehen, was zu Lecks führt. Oder das falsche Öl oder den falschen Filter zu verwenden. Solche Tipps wären hilfreich.\n\nJetzt überprüfe ich, ob jeder Schritt klar und in der richtigen Reihenfolge ist. Beginne mit dem Parken und der Sicherheit, sammle die Werkzeuge ein, lasse das Öl ab, ersetze den Filter, fülle Öl nach, prüfe auf Lecks und entsorge das alte Öl. Ja, das ergibt Sinn. Ich erinnere mich auch daran, den Nutzer zu bitten, das Handbuch des Fahrzeugs zu konsultieren. Alles in allem, das sollte die wichtigsten Punkte abdecken. Zeit, alles zusammenzustellen, in eine schrittweise Anleitung, mit etwas Fantasie gewürzt."
}
]

In [43]:
# Setup SFTTrainer for training
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=1,  # Full training run
        # max_steps=100,  # Uncomment for quick test
        learning_rate=2e-4,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        save_steps=500,
        save_total_limit=2,
        report_to="none",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
    ),
)

print("Trainer initialized successfully!")

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/2684 [00:00<?, ? examples/s]

Trainer initialized successfully!


In [44]:
# Apply train_on_responses_only to only train on assistant outputs
# This ignores loss on user inputs and improves accuracy
from unsloth.chat_templates import train_on_responses_only

# GPT-OSS specific kwargs for response masking
gpt_oss_kwargs = dict(
    instruction_part="<|start|>user<|message|>", 
    response_part="<|start|>assistant<|channel|>final<|message|>"
)

trainer = train_on_responses_only(
    trainer,
    **gpt_oss_kwargs,
)

print("Applied train_on_responses_only - will only train on assistant outputs")

Map (num_proc=16):   0%|          | 0/2684 [00:00<?, ? examples/s]

Applied train_on_responses_only - will only train on assistant outputs


In [38]:
# Verify masking is working correctly
print("=== Verifying Label Masking ===\n")
print("Original input (first 1000 chars):")
print(tokenizer.decode(trainer.train_dataset[0]["input_ids"])[:1000])
print("\n" + "="*50 + "\n")
print("Masked labels (what model trains on - first 1000 chars):")
masked = [tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[0]["labels"]]
print(tokenizer.decode(masked).replace(tokenizer.pad_token, " ")[:1000])

=== Verifying Label Masking ===

Original input (first 1000 chars):
<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2025-12-30

Reasoning: medium

# Valid channels: analysis, commentary, final. Channel must be included for every message.
Calls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>developer<|message|># Instructions

You are a mathematical reasoning assistant. Solve the problem step by step and provide the final numerical answer.<|end|><|start|>user<|message|>In an acute scalene triangle $ABC$, points $D,E,F$ lie on sides $BC, CA, AB$, respectively, such that $AD \perp BC, BE \perp CA, CF \perp AB$. Altitudes $AD, BE, CF$ meet at orthocenter $H$. Points $P$ and $Q$ lie on segment $EF$ such that $AP \perp EF$ and $HQ \perp EF$. Lines $DP$ and $QH$ intersect at point $R$. Compute $HQ/HR$.<|end|><|start|>assistant<|channel|>analysis<|message|>
In an acute scalene triangle \(

In [45]:
# Show GPU memory stats before training
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved before training.")

GPU = NVIDIA L4. Max memory = 22.161 GB.
12.811 GB of memory reserved before training.


In [ ]:
# ========================================
# BEFORE TRAINING: Inference on Validation Set
# ========================================
import re
from tqdm import tqdm

def extract_final_answer(text):
    """Extract integer from model output"""
    # Look for the final answer after the last channel marker
    m = re.search(r"-?\d+", text.split("<|channel|>final<|message|>")[-1] if "<|channel|>final<|message|>" in text else text)
    return int(m.group(0)) if m else None

def run_validation_inference(model, tokenizer, val_df, desc="Inference"):
    """Run inference on validation set and return predictions + ground truth"""
    model.eval()
    results = []
    
    for idx, row in tqdm(val_df.iterrows(), total=len(val_df), desc=desc):
        problem = row.get("Problem") or row.get("problem")
        gt_answer = int(re.search(r"-?\d+", str(row.get("final_answer") or row.get("clean_answer") or row.get("answer"))).group(0))
        
        messages = [
            {"role": "system", "content": "You are a mathematical reasoning assistant. Solve the problem step by step and provide the final numerical answer."},
            {"role": "user", "content": str(problem)},
        ]
        
        inputs = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
            reasoning_effort="medium",
        ).to("cuda")
        
        with torch.no_grad():
            output = model.generate(**inputs, max_new_tokens=2048, do_sample=False)
        
        response = tokenizer.decode(output[0], skip_special_tokens=False)
        pred_answer = extract_final_answer(response)
        
        results.append({
            "idx": idx,
            "ground_truth": gt_answer,
            "prediction": pred_answer,
            "correct": pred_answer == gt_answer if pred_answer is not None else False,
            "response": response[-500:]  # Keep last 500 chars for debugging
        })
    
    return results

print("Running BEFORE training inference...")
before_results = run_validation_inference(model, tokenizer, Val_df, desc="Before Training")

before_correct = sum(r["correct"] for r in before_results)
before_accuracy = before_correct / len(before_results) * 100
print(f"\n=== BEFORE TRAINING ===")
print(f"Correct: {before_correct}/{len(before_results)}")
print(f"Accuracy: {before_accuracy:.1f}%")

In [46]:
# Start training!
print(f"Starting training on {len(dataset)} examples...")
print(f"LoRA rank: 16, Target modules: q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj")
print("="*50)

trainer_stats = trainer.train()

Starting training on 2684 examples...
LoRA rank: 16, Target modules: q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,684 | Num Epochs = 1 | Total steps = 168
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 7,962,624 of 20,922,719,808 (0.04% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,4.213500
20,0.237000
30,0.121600
40,0.212100
50,0.325700
60,0.269200
70,0.133600
80,0.163400
90,0.180300
100,0.133200


In [47]:
# Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)

print("="*50)
print("TRAINING COMPLETE!")
print("="*50)
print(f"Training time: {trainer_stats.metrics['train_runtime']:.2f} seconds")
print(f"Training time: {trainer_stats.metrics['train_runtime']/60:.2f} minutes")
print(f"Peak reserved memory = {used_memory} GB")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB")
print(f"Peak reserved memory % of max = {used_percentage}%")
print(f"Peak training memory % of max = {lora_percentage}%")

TRAINING COMPLETE!
Training time: 2786.32 seconds
Training time: 46.44 minutes
Peak reserved memory = 18.74 GB
Peak reserved memory for training = 5.929 GB
Peak reserved memory % of max = 84.563%
Peak training memory % of max = 26.754%


In [53]:
# ========================================
# AFTER TRAINING: Inference on Validation Set
# ========================================
print("Running AFTER training inference...")
after_results = run_validation_inference(model, tokenizer, Val_df, desc="After Training")

after_correct = sum(r["correct"] for r in after_results)
after_accuracy = after_correct / len(after_results) * 100
print(f"\n=== AFTER TRAINING ===")
print(f"Correct: {after_correct}/{len(after_results)}")
print(f"Accuracy: {after_accuracy:.1f}%")

Running AFTER training inference...


After Training:   5%|▌         | 1/20 [01:59<37:54, 119.69s/it]


KeyboardInterrupt: 

In [ ]:
# ========================================
# COMPARISON: Before vs After Training
# ========================================
import matplotlib.pyplot as plt

# Create comparison DataFrame
comparison_df = []
for b, a in zip(before_results, after_results):
    comparison_df.append({
        "Sample": b["idx"],
        "Ground Truth": b["ground_truth"],
        "Before Pred": b["prediction"],
        "After Pred": a["prediction"],
        "Before Correct": b["correct"],
        "After Correct": a["correct"],
    })

import pandas as pd
comparison_df = pd.DataFrame(comparison_df)

# Print detailed results
print("="*70)
print("DETAILED RESULTS")
print("="*70)
print(comparison_df.to_string(index=False))

# Summary stats
print("\n" + "="*70)
print("SUMMARY")
print("="*70)
print(f"Before Training Accuracy: {before_accuracy:.1f}% ({before_correct}/{len(before_results)})")
print(f"After Training Accuracy:  {after_accuracy:.1f}% ({after_correct}/{len(after_results)})")
print(f"Improvement: {after_accuracy - before_accuracy:+.1f}%")

# Count changes
improved = sum(1 for b, a in zip(before_results, after_results) if not b["correct"] and a["correct"])
regressed = sum(1 for b, a in zip(before_results, after_results) if b["correct"] and not a["correct"])
stayed_correct = sum(1 for b, a in zip(before_results, after_results) if b["correct"] and a["correct"])
stayed_wrong = sum(1 for b, a in zip(before_results, after_results) if not b["correct"] and not a["correct"])

print(f"\nImproved (wrong→correct): {improved}")
print(f"Regressed (correct→wrong): {regressed}")
print(f"Stayed correct: {stayed_correct}")
print(f"Stayed wrong: {stayed_wrong}")

In [ ]:
# ========================================
# PLOTS
# ========================================
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Plot 1: Bar chart - Before vs After Accuracy
ax1 = axes[0]
bars = ax1.bar(["Before Training", "After Training"], [before_accuracy, after_accuracy], 
               color=["#ff6b6b", "#4ecdc4"], edgecolor="black", linewidth=1.5)
ax1.set_ylabel("Accuracy (%)", fontsize=12)
ax1.set_title("Validation Accuracy: Before vs After", fontsize=14, fontweight="bold")
ax1.set_ylim(0, 100)
for bar, val in zip(bars, [before_accuracy, after_accuracy]):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, f"{val:.1f}%", 
             ha="center", fontsize=12, fontweight="bold")

# Plot 2: Per-sample comparison
ax2 = axes[1]
x = range(len(before_results))
before_correct_list = [1 if r["correct"] else 0 for r in before_results]
after_correct_list = [1 if r["correct"] else 0 for r in after_results]
width = 0.35
ax2.bar([i - width/2 for i in x], before_correct_list, width, label="Before", color="#ff6b6b", alpha=0.8)
ax2.bar([i + width/2 for i in x], after_correct_list, width, label="After", color="#4ecdc4", alpha=0.8)
ax2.set_xlabel("Sample Index", fontsize=12)
ax2.set_ylabel("Correct (1) / Wrong (0)", fontsize=12)
ax2.set_title("Per-Sample Correctness", fontsize=14, fontweight="bold")
ax2.legend()
ax2.set_xticks(x)

# Plot 3: Pie chart - Change distribution
ax3 = axes[2]
labels = ["Improved", "Regressed", "Stayed Correct", "Stayed Wrong"]
sizes = [improved, regressed, stayed_correct, stayed_wrong]
colors = ["#4ecdc4", "#ff6b6b", "#95e1d3", "#f38181"]
explode = (0.05, 0.05, 0, 0)
ax3.pie(sizes, explode=explode, labels=labels, colors=colors, autopct="%1.0f%%",
        shadow=True, startangle=90)
ax3.set_title("Training Impact Distribution", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.savefig("training_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("\nPlot saved to 'training_comparison.png'")

In [48]:
# Test inference with a sample problem
from transformers import TextStreamer

sample_problem = Training_df.iloc[0]["Problem"]

messages = [
    {
        "role": "system",
        "content": "You are a mathematical assistant. Output ONLY the final integer answer.",
    },
    {"role": "user", "content": sample_problem},
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
    reasoning_effort="medium",
).to("cuda")

print("Sample Problem:")
print(sample_problem[:500] + "..." if len(sample_problem) > 500 else sample_problem)
print("\n" + "="*50 + "\n")
print("Model Response (should be just an integer):")
_ = model.generate(**inputs, max_new_tokens=64, streamer=TextStreamer(tokenizer))

Sample Problem:
In an acute scalene triangle $ABC$, points $D,E,F$ lie on sides $BC, CA, AB$, respectively, such that $AD \perp BC, BE \perp CA, CF \perp AB$. Altitudes $AD, BE, CF$ meet at orthocenter $H$. Points $P$ and $Q$ lie on segment $EF$ such that $AP \perp EF$ and $HQ \perp EF$. Lines $DP$ and $QH$ intersect at point $R$. Compute $HQ/HR$.


Model Response (should be just an integer):
<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2025-12-30

Reasoning: medium

# Valid channels: analysis, commentary, final. Channel must be included for every message.
Calls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>developer<|message|># Instructions

You are a mathematical assistant. Output ONLY the final integer answer.<|end|><|start|>user<|message|>In an acute scalene triangle $ABC$, points $D,E,F$ lie on sides $BC, CA, AB$, respectively, such that $AD \perp BC, BE \perp CA, CF \pe

In [49]:
# Save the finetuned model (LoRA adapters)
model.save_pretrained("gpt_oss_20b_finetuned_lora")
tokenizer.save_pretrained("gpt_oss_20b_finetuned_lora")
print("Model saved to 'gpt_oss_20b_finetuned_lora'")

Model saved to 'gpt_oss_20b_finetuned_lora'


In [52]:
handle = 'barnobarno/aimo-oss-finetuned-20b-LORA'
local_dataset_dir = 'gpt_oss_20b_finetuned_lora'
kagglehub.dataset_upload(handle, local_dataset_dir)

# You can then create a new version of this existing dataset and include version notes (optional).
kagglehub.dataset_upload(handle, local_dataset_dir, version_notes='improved data')


Uploading Dataset https://www.kaggle.com/datasets/barnobarno/aimo-oss-finetuned-20b-LORA ...
Starting upload for file gpt_oss_20b_finetuned_lora/adapter_model.safetensors


Uploading: 100%|██████████| 31.9M/31.9M [00:03<00:00, 9.41MB/s]

Upload successful: gpt_oss_20b_finetuned_lora/adapter_model.safetensors (30MB)
Starting upload for file gpt_oss_20b_finetuned_lora/special_tokens_map.json



Uploading: 100%|██████████| 446/446 [00:00<00:00, 561B/s]

Upload successful: gpt_oss_20b_finetuned_lora/special_tokens_map.json (446B)
Starting upload for file gpt_oss_20b_finetuned_lora/adapter_config.json


...


Uploading: 100%|██████████| 5.25k/5.25k [00:00<00:00, 6.77kB/s]

Upload successful: gpt_oss_20b_finetuned_lora/README.md (5KB)
Starting upload for file gpt_oss_20b_finetuned_lora/tokenizer.json



Uploading: 100%|██████████| 27.9M/27.9M [00:03<00:00, 8.50MB/s]

Upload successful: gpt_oss_20b_finetuned_lora/tokenizer.json (27MB)


Your dataset has been created.
Files are being processed...
See at: https://www.kaggle.com/datasets/barnobarno/aimo-oss-finetuned-20b-LORA


In [ ]:
# Optional: Save merged model in different formats
# Uncomment the format you want to use

# Save as merged 16-bit model
# model.save_pretrained_merged("gpt_oss_20b_merged_16bit", tokenizer, save_method="merged_16bit")

# Save as MXFP4 format
# model.save_pretrained_merged("gpt_oss_20b_merged_mxfp4", tokenizer, save_method="mxfp4")

# Push to Hugging Face Hub (uncomment and add your token)
# model.push_to_hub("your_username/gpt_oss_20b_finetuned", token="hf_...")
# tokenizer.push_to_hub("your_username/gpt_oss_20b_finetuned", token="hf_...")

print("To push to HuggingFace Hub, uncomment the push_to_hub lines above and add your token")

## Loading the Finetuned Model

To load and use the finetuned model later:

```python
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "gpt_oss_20b_finetuned_lora",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

# Use for inference
messages = [
    {"role": "system", "content": "You are a mathematical reasoning assistant. Solve the problem step by step and provide the final numerical answer."},
    {"role": "user", "content": "Your math problem here"},
]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True, reasoning_effort="high").to("cuda")
output = model.generate(**inputs, max_new_tokens=2048)
```